## Boilerplate

This notebook is a replication of the (Markowitz et. al., 2023) paper. Specifically called:
*Spontaneous behaviour is structured by reinforcement without explicit reward*

In [1]:
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()

# Find the repo dir named 'syllable_generation'
for p in (start, *start.parents):
    if p.name == "syllable_generation":
        REPO_ROOT = p
        break
else:
    raise RuntimeError("Could not find 'syllable_generation'")

# Make the package importable: add the *parent* of the repo to sys.path
PKG_PARENT = REPO_ROOT.parent
for pth in (str(PKG_PARENT), str(REPO_ROOT)):  # both, for convenience
    if pth not in sys.path:
        sys.path.insert(0, pth)

# optional sanity check:
import importlib; importlib.import_module("syllable_generation")

<module 'syllable_generation' from '/home/fsp585/Desktop/FiberphotometryCode/syllable_generation/__init__.py'>

In [2]:
# --- user-editable constants ---
PATH_PREFIX = "/home/fsp585/Desktop/FiberphotometryCode/syllable_generation/"
SESSIONS_PKL = PATH_PREFIX + "loaders/jonathan_ifdn_sessions_w_sylls.pkl"
RESULTS_PATH = PATH_PREFIX + "syllabizers/keypoint_moseq_syll/demo_project/full-run/checkpoint.h5"

PROJECT_DIR = PATH_PREFIX + "syllabizers/keypoint_moseq_syll/demo_project"
MODEL_NAME = "full-run"

USE_CPU = True           # set True to avoid reserving GPU for loading KPMS
OVERWRITE_PICKLE = True  # if False and file exists, we won't overwrite
PRINT_SAMPLE = True      # print a couple sample heads for inspection

In [3]:
import pandas as pd

In [4]:
sessions = pd.read_pickle(SESSIONS_PKL)

## Setup

We setup syllable data, kinematic data and fiberphotometry data

In [5]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"  # avoid grabbing the GPU for read-only ops

import keypoint_moseq as kpms

In [6]:
moseq_df = kpms.compute_moseq_df(PROJECT_DIR, MODEL_NAME, smooth_heading=True)

In [7]:
stats_df = kpms.compute_stats_df(
    PROJECT_DIR,
    MODEL_NAME,
    moseq_df,
    min_frequency=0.005,  # threshold frequency for including a syllable in the dataframe
    groupby=["group", "name"],  # column(s) to group the dataframe by
)  # frame rate of the video from which keypoints were inferred

In [8]:
from syllable_generation.analysis.replications.datta_utils import combined_photometry_and_syllables, build_fixed_moseq_dfs,\
      build_aligned_moseq_phot

In [9]:
sessions[0].photometry['470']

,FrameCounter,SystemTimestamp,LedState,ComputerTimestamp,signal_0,signal_1,signal_2,signal_3,frame_idx,signal_0_dff,...,signal_0_zscore_datta,signal_1_dff,signal_1_zscore,signal_1_zscore_datta,signal_2_dff,signal_2_zscore,signal_2_zscore_datta,signal_3_dff,signal_3_zscore,signal_3_zscore_datta
0,1309,1256.156000,2,4.442315e+07,0.003923,0.003922,0.004299,0.003924,0,0.000331,...,0.724187,-0.35743,0.548259,-0.369905,0.096154,1.137778,1.360219,-0.442062,0.621173,0.265515
1,1313,1256.206016,2,4.442320e+07,0.003922,0.003922,0.004262,0.003922,4,0.0,...,-0.813258,-0.361523,0.499462,-0.363266,0.086859,1.096811,-0.28345,-0.443049,0.611098,-1.092284
2,1317,1256.256000,2,4.442325e+07,0.003922,0.003922,0.004255,0.003935,8,0.0,...,-0.809921,-0.363214,0.479296,-0.367556,0.084936,1.088335,-0.628777,-0.442687,0.614792,1.921745
3,1321,1256.306016,2,4.442330e+07,0.003922,0.003922,0.004236,0.003929,12,0.0,...,-0.806624,-0.360609,0.510359,-0.366109,0.080128,1.067145,-1.4801,-0.443965,0.601743,0.220087
4,1325,1256.356000,2,4.442335e+07,0.003922,0.003922,0.004279,0.003924,16,0.0,...,-0.803368,-0.362246,0.490838,-0.366832,0.091026,1.115175,0.41972,-0.442465,0.617055,-0.875427
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79781,320433,5245.194080,2,4.841255e+07,0.003922,0.003922,0.003961,0.003922,319124,0.0,...,-0.06984,-0.124886,3.320627,-0.242757,-0.002848,0.701428,0.133336,-0.219962,2.888455,-1.155073
79782,320437,5245.244064,2,4.841260e+07,0.003922,0.003922,0.003953,0.003922,319128,0.0,...,-0.070011,-0.124202,3.328786,0.152682,-0.003012,0.700708,-0.871964,-0.215094,2.938151,-0.452231
79783,320441,5245.294080,2,4.841265e+07,0.003922,0.003922,0.003962,0.003922,319132,0.0,...,-0.070183,-0.126366,3.302981,0.233801,0.000793,0.717477,0.32817,-0.213839,2.950962,-0.042551
79784,320445,5245.344064,2,4.841270e+07,0.003922,0.003922,0.003957,0.003922,319136,0.0,...,-0.070357,-0.126819,3.297581,-0.241537,-0.001269,0.708388,-0.355216,-0.217932,2.909183,-0.276733


In [10]:
combined_photometry_and_syllables(sessions[0])

,SystemTimestamp,signal_0,signal_1,signal_2,signal_3,signal_0_dff,signal_0_zscore,signal_0_zscore_datta,signal_1_dff,signal_1_zscore,signal_1_zscore_datta,signal_2_dff,signal_2_zscore,signal_2_zscore_datta,signal_3_dff,signal_3_zscore,signal_3_zscore_datta
0,1256.156000,0.003923,0.003922,0.004299,0.003924,0.000331,1.295507,0.724187,-0.357430,0.548259,-0.369905,0.096154,1.137778,1.360219,-0.442062,0.621173,0.265515
1,1256.168504,0.003923,0.003922,0.004290,0.003924,0.000248,1.294632,0.339826,-0.358453,0.536060,-0.368245,0.093830,1.127536,0.949302,-0.442309,0.618654,-0.073935
2,1256.181008,0.003922,0.003922,0.004280,0.003923,0.000165,1.293757,-0.044535,-0.359476,0.523861,-0.366585,0.091506,1.117294,0.538384,-0.442555,0.616136,-0.413384
3,1256.193512,0.003922,0.003922,0.004271,0.003922,0.000083,1.292882,-0.428896,-0.360500,0.511661,-0.364925,0.089183,1.107053,0.127467,-0.442802,0.613617,-0.752834
4,1256.206016,0.003922,0.003922,0.004262,0.003922,0.000000,1.292007,-0.813258,-0.361523,0.499462,-0.363266,0.086859,1.096811,-0.283450,-0.443049,0.611098,-1.092284
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319139,5245.381576,0.003922,0.003922,0.003964,0.003922,0.000000,1.292007,-0.070488,-0.123826,3.333266,-0.973504,-0.000437,0.712056,0.652387,-0.218185,2.906603,-0.776469
319140,5245.394080,0.003922,0.003922,0.003967,0.003922,0.000000,1.292007,-0.070532,-0.122828,3.345161,-1.217494,-0.000160,0.713278,0.988255,-0.218269,2.905743,-0.943048
319141,5245.394080,0.003922,0.003922,0.003967,0.003922,0.000000,1.292007,-0.070532,-0.122828,3.345161,-1.217494,-0.000160,0.713278,0.988255,-0.218269,2.905743,-0.943048
319142,5245.394080,0.003922,0.003922,0.003967,0.003922,0.000000,1.292007,-0.070532,-0.122828,3.345161,-1.217494,-0.000160,0.713278,0.988255,-0.218269,2.905743,-0.943048


In [11]:
kpms.compute_moseq_df(PROJECT_DIR, MODEL_NAME, smooth_heading=True)

,name,centroid_x,centroid_y,heading,angular_velocity,velocity_px_s,syllable,frame_index,group,onset
0,Arena1234_CenterTopCamVid_2024-12-13T12_20_06_...,167.051823,199.522437,-1.122018,0.000000,0.000000,56,0,WT,True
1,Arena1234_CenterTopCamVid_2024-12-13T12_20_06_...,166.903142,199.062367,-1.122018,0.012100,14.504940,56,1,WT,False
2,Arena1234_CenterTopCamVid_2024-12-13T12_20_06_...,167.223306,199.104034,-1.122018,0.027302,9.685929,56,2,WT,False
3,Arena1234_CenterTopCamVid_2024-12-13T12_20_06_...,167.995061,199.575542,-1.136146,0.046672,27.131755,56,3,WT,False
4,Arena1234_CenterTopCamVid_2024-12-13T12_20_06_...,168.522242,201.069801,-1.122018,0.068494,47.535846,56,4,WT,False
...,...,...,...,...,...,...,...,...,...,...
4114930,Arena1234_CenterTopCamVid_2024-12-17T12_37_40_...,70.409914,128.542315,-1.604544,1.043472,86.171672,54,143807,UNK,False
4114931,Arena1234_CenterTopCamVid_2024-12-17T12_37_40_...,68.178179,126.800341,-1.516835,0.849673,84.932943,54,143808,UNK,False
4114932,Arena1234_CenterTopCamVid_2024-12-17T12_37_40_...,66.674209,124.029043,-1.488353,0.645352,94.592898,54,143809,UNK,False
4114933,Arena1234_CenterTopCamVid_2024-12-17T12_37_40_...,66.683953,121.307837,-1.488353,0.433392,81.636705,54,143810,UNK,False


In [12]:
fixed_moseq_dfs = build_fixed_moseq_dfs(sessions, moseq_df, PROJECT_DIR, MODEL_NAME)

In [13]:
all_data_dfs = build_aligned_moseq_phot(sessions, fixed_moseq_dfs)
for s_idx, s in enumerate(sessions):
    s.all_data_df = all_data_dfs[s_idx]

In [14]:
import pickle
from pathlib import Path

out_path = Path("/home/fsp585/Desktop/FiberphotometryCode/syllable_generation/loaders/jonathan_ifdn_sessions_w_sylls.pkl")
out_path.parent.mkdir(parents=True, exist_ok=True)

with out_path.open("wb") as f:
    pickle.dump(sessions, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved {len(sessions)} sessions → {out_path}")


Saved 19 sessions → /home/fsp585/Desktop/FiberphotometryCode/syllable_generation/loaders/jonathan_ifdn_sessions_w_sylls.pkl


In [15]:
for s in sessions:
    print(len(s.all_data_df))

221392
217043
230326
171509
221597
218981
218870
221039
219419
217678
251911
216672
236585
225843
218628
222436
221143
220051
143812
